# Modul B · Kapitel 4 — Model Cards und Speicher

> 🛠️ **Workshop-Version:** Bearbeite die zwei markierten Aufgaben.

**Lernziel:** Du kannst abschätzen, ob ein Modell auf eine Hardware passt.

Dieses Notebook folgt einem kurzen Pfad: Begriff verstehen → Rechnung oder
Messung durchführen → Ergebnis für eine Deployment-Entscheidung nutzen.
Programmiert werden nur zwei Kernstellen: Gewichtsspeicher und ein vollständiges VRAM-Budget. Hilfs- und
Visualisierungscode ist bewusst vorgegeben.


## 0 · Setup

Die nächsten Zellen laden Hilfsfunktionen, Model Cards, Hardwaredaten und Szenarien.


In [ ]:
import sys
from pathlib import Path

# helfer.py liegt neben dem Notebook. Der Suchlauf findet es auch, wenn das
# Arbeitsverzeichnis woanders liegt — etwa in Colab.
for kandidat in [Path.cwd(), Path.cwd() / "04_deployment", *Path.cwd().parents]:
    if (kandidat / "helfer.py").exists():
        sys.path.insert(0, str(kandidat))
        break

try:
    import matplotlib
    import pandas
except ImportError:
    %pip install -q matplotlib pandas openai tiktoken
    import matplotlib

import re
import statistics

import matplotlib.pyplot as plt
import numpy as np

import helfer
from helfer import GB, lade_daten, messe_anfrage, zeige_tabelle

# Einheitliche Farben für alle Diagramme in diesem Notebook
BLAU, ORANGE, TEAL, GRAU = "#2563eb", "#e8590c", "#0d9488", "#6b7280"

plt.rcParams.update({
    "figure.figsize": (9, 4.5),
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": GRAU,
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": "#e5e7eb",
    "grid.linewidth": 0.8,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "font.size": 10,
})

print(f"GB = {GB:,} Byte".replace(",", "."))
print(f"Modell für die Messungen: {helfer.MODELL} über {helfer.BASIS_URL}")
print("Setup fertig ✔")

In [ ]:
# ▶️ Die drei Datendateien
KARTEN_DATEI = lade_daten("model_cards")
MODELLE = KARTEN_DATEI["modelle"]
HARDWARE = lade_daten("hardware")["karten"]
SZENARIEN = lade_daten("szenarien")["szenarien"]

MODELL_NACH_NAME = {m["name"]: m for m in MODELLE}
KARTE_NACH_NAME = {k["name"]: k for k in HARDWARE}

print(f"{len(MODELLE)} Model Cards, {len(HARDWARE)} Karten, "
      f"{len(SZENARIEN)} Szenarien   (Stand {KARTEN_DATEI['stand']})")
print()

zeige_tabelle([{
    "Modell": m["name"],
    "Typ": m["typ"],
    "Parameter": f"{m['parameter_gesamt'] / 1e9:.2f} Mrd.",
    "aktiv": f"{m['parameter_aktiv'] / 1e9:.2f} Mrd." if m["parameter_aktiv"] else "—",
    "Schichten": m["schichten"],
    "KV-Köpfe": m["kv_koepfe"],
    "Kopf-Dim.": m["kopf_dimension"],
    "Kontext": f"{m['kontextlaenge'] // 1024}k",
    "Lizenz": m["lizenz"],
} for m in MODELLE])

## 1 · Was der Modellname verrät

Modellnamen nennen meist Familie, ungefähre Parameterzahl und Variante. Für belastbare Werte gilt trotzdem die Model Card.


In [ ]:
# ▶️ Die Bausteine für den nächsten Schritt — fertig, du brauchst sie gleich
GROESSE = re.compile(r"^(\d+(?:[.,]\d+)?)([MB])$", re.IGNORECASE)         # 8B, 135M, 1.5B
AKTIV = re.compile(r"^([AE])(\d+(?:[.,]\d+)?)([MB])$", re.IGNORECASE)     # A3B, E4B
QUANT = re.compile(r"^(I?Q\d\S*|BF16|FP16|FP8|INT8|INT4|GGUF|AWQ|GPTQ|MLX)$", re.IGNORECASE)
ROLLEN = {"base", "pt", "it", "instruct", "chat", "assistant",
          "coder", "code", "math", "vision", "reasoning", "thinking"}


def zahl(betrag, einheit):
    """'1.5', 'B' → 1500000000.0 — die Größenangabe als absolute Parameterzahl."""
    return float(betrag.replace(",", ".")) * {"M": 10**6, "B": 10**9}[einheit.upper()]


print(f"{'Segment':<10} {'Größe':<7} {'A/E':<7} {'Quant.':<7} {'Rolle':<7}")
for teil in ["Llama", "8B", "135M", "A3B", "E4B", "Instruct", "Q4_K_M"]:
    print(f"  {teil:<10} {str(bool(GROESSE.match(teil))):<7} "
          f"{str(bool(AKTIV.match(teil))):<7} "
          f"{str(bool(QUANT.match(teil))):<7} "
          f"{str(teil.lower() in ROLLEN):<7}")

In [ ]:
# Vorgegebene Hilfsfunktion
def lies_modellnamen(name):
    """Zerlegt einen Modellnamen in Hersteller, Familie, Größe, Typ, Rollen, Quantisierung."""
    hersteller = None
    if "/" in name:
        hersteller, name = name.split("/", 1)

    familie = []
    parameter, aktiv, moe = None, None, False
    rollen, quantisierung = [], None

    for teil in name.split("-"):
        treffer = GROESSE.match(teil)
        if treffer:
            parameter = zahl(treffer.group(1), treffer.group(2))
            continue

        treffer = AKTIV.match(teil)
        if treffer:
            aktiv = zahl(treffer.group(2), treffer.group(3))
            moe = treffer.group(1).upper() == "A"   # E steht für effektive Parameter
            continue

        if QUANT.match(teil):
            quantisierung = teil
            continue

        if teil.lower() in ROLLEN:
            rollen.append(teil.lower())
            continue

        # Alles Übrige gehört zur Familie — aber nur vor der Größe und vor der ersten Rolle.
        if parameter is None and aktiv is None and not rollen:
            familie.append(teil)

    return {
        "hersteller": hersteller,
        "familie": "-".join(familie),
        "parameter_gesamt": parameter,
        "parameter_aktiv": aktiv,
        "typ": "moe" if moe else "dense",
        "rollen": rollen,
        "quantisierung": quantisierung,
    }

## 2 · Gewichtsspeicher

Die Kernrechnung lautet: `Parameter × Bits ÷ 8`. Eine Milliarde Parameter benötigt in FP16 etwa 2 GB, in INT8 1 GB und in INT4 0,5 GB.


### 🛠️ Aufgabe 1 — Gewichtsspeicher berechnen

Implementiere `speicher_gewichte(parameter, bits)`. Gib den Speicher in GB zurück. Nutze `GB = 10**9` und die Umrechnung von Bit zu Byte.

Führe danach den Selbsttest aus. Die ausgefüllte Variante steht in der Lösungsversion des Notebooks.


In [ ]:
def speicher_gewichte(parameter, bits):
    """Speicher für die Gewichte in GB (GB = 10^9 Byte)."""
    # TODO: Ersetze die nächste Zeile
    raise NotImplementedError("Aufgabe 1: speicher_gewichte() implementieren")


In [ ]:
# ✅ Selbsttest
assert speicher_gewichte(1e9, 16) == 2.0, "1 Mrd. Parameter à 16 Bit sind 2 GB"
assert speicher_gewichte(1e9, 8) == 1.0
assert speicher_gewichte(1e9, 4) == 0.5
assert speicher_gewichte(0, 16) == 0.0
assert abs(speicher_gewichte(70.6e9, 4) - 35.3) < 1e-6, "Llama 3.1 70B in INT4"
assert abs(speicher_gewichte(8.03e9, 16) - 16.06) < 1e-6, "Llama 3.1 8B in FP16"
assert speicher_gewichte(7e9, 8) == 2 * speicher_gewichte(7e9, 4), "Halbe Bits, halber Speicher"
print("✅ Aufgabe 1 gelöst")
print()
print(f"Llama 3.1 8B  in FP16: {speicher_gewichte(8.03e9, 16):5.2f} GB")
print(f"Llama 3.1 8B  in INT4: {speicher_gewichte(8.03e9, 4):5.2f} GB")
print(f"Llama 3.1 70B in INT4: {speicher_gewichte(70.6e9, 4):5.2f} GB")

In [ ]:
# ▶️ Alle Modelle in allen drei Präzisionen
BITS = {"FP16": 16, "INT8": 8, "INT4": 4}

zeige_tabelle([{
    "Modell": m["name"].split("/")[-1],
    "Parameter": f"{m['parameter_gesamt'] / 1e9:.2f} Mrd.",
    **{f"{p} (GB)": speicher_gewichte(m["parameter_gesamt"], b) for p, b in BITS.items()},
    "Gewichte unter 24 GB": ", ".join(
        p for p, b in BITS.items()
        if speicher_gewichte(m["parameter_gesamt"], b) < 24) or "—",
} for m in MODELLE])

In [ ]:
# ▶️ Dasselbe als Bild. Die Achse ist logarithmisch — sonst verschwindet SmolLM2 neben 70B.
namen = [m["name"].split("/")[-1] for m in MODELLE]
x = np.arange(len(MODELLE))
breite = 0.27

fig, achse = plt.subplots(figsize=(11, 4.6))
for i, (praezision, bits) in enumerate(BITS.items()):
    werte = [speicher_gewichte(m["parameter_gesamt"], bits) for m in MODELLE]
    achse.bar(x + (i - 1) * breite, werte, breite, label=praezision,
              color=[BLAU, TEAL, ORANGE][i])

for grenze, text in ((8, "8 GB · RTX 4060 Laptop"), (24, "24 GB · RTX 4090"),
                     (80, "80 GB · A100 / H100")):
    achse.axhline(grenze, color=GRAU, linewidth=1.0, linestyle="--")
    achse.text(len(MODELLE) - 0.4, grenze * 1.06, text, ha="right", fontsize=9, color=GRAU)

achse.set_yscale("log")
achse.set_xticks(x)
achse.set_xticklabels(namen, rotation=35, ha="right", fontsize=9)
achse.set_ylabel("Speicher für die Gewichte (GB)")
achse.set_title("Was die Gewichte kosten")
achse.legend(frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
# ▶️ Wie groß ist ein Q4_K_M-Modell wirklich? Die Zahl kommt vom lokalen Ollama-Dienst.
def ollama_groesse_gb(modell):
    """Größe eines Modells auf der Platte in GB, gemeldet vom lokalen Ollama-Dienst."""
    import json
    import urllib.request

    adresse = helfer.BASIS_URL.replace("/v1", "") + "/api/tags"
    try:
        with urllib.request.urlopen(adresse, timeout=5) as antwort:
            eintraege = json.load(antwort)["models"]
    except Exception as fehler:
        print(f"Ollama nicht erreichbar ({type(fehler).__name__}) — Vergleich entfällt.")
        return None

    for eintrag in eintraege:
        if eintrag["name"].split(":")[0] == modell.split(":")[0]:
            return eintrag["size"] / GB
    return None


# Der Ollama-Eintrag `llama3.2` ist Llama 3.2 3B Instruct, quantisiert als Q4_K_M.
# Für diese Gegenprobe braucht es beides: die Model Card und die fertige Datei.
LOKAL = MODELL_NACH_NAME["meta-llama/Llama-3.2-3B-Instruct"]
gemessen = ollama_groesse_gb("llama3.2")

print(f"Parameter laut Card:      {LOKAL['parameter_gesamt'] / 1e9:.2f} Mrd.")
print(f"nominell mit 4,0 Bit:     {speicher_gewichte(LOKAL['parameter_gesamt'], 4.0):5.2f} GB")
print(f"gerechnet mit 4,7 Bit:    {speicher_gewichte(LOKAL['parameter_gesamt'], 4.7):5.2f} GB")
if gemessen:
    bits_echt = gemessen * GB * 8 / LOKAL["parameter_gesamt"]
    print(f"tatsächliche Dateigröße:  {gemessen:5.2f} GB   → {bits_echt:.2f} Bit je Parameter")

## 3 · Dense und Mixture of Experts

Bei Dense-Modellen sind alle Parameter aktiv. Bei MoE-Modellen bestimmt die Gesamtzahl den Speicher, die aktive Zahl vor allem den Rechenaufwand.


In [ ]:
# ▶️ MoE gegen ein Dense-Modell derselben Klasse
def gegenueberstellung(modell, bits=4):
    gesamt = modell["parameter_gesamt"]
    aktiv = modell["parameter_aktiv"] or gesamt
    return {
        "Modell": modell["name"].split("/")[-1],
        "Typ": modell["typ"],
        "Parameter gesamt": f"{gesamt / 1e9:.1f} Mrd.",
        "Parameter aktiv": f"{aktiv / 1e9:.1f} Mrd.",
        "Speicher INT4 (GB)": speicher_gewichte(gesamt, bits),
        "gelesen je Token (GB)": speicher_gewichte(aktiv, bits),
        "Anteil aktiv": f"{aktiv / gesamt * 100:.0f} %",
    }


zeige_tabelle([gegenueberstellung(MODELL_NACH_NAME[n]) for n in [
    "mistralai/Mixtral-8x7B-Instruct-v0.1",
    "Qwen/Qwen3-30B-A3B",
    "meta-llama/Llama-3.1-70B-Instruct",
    "meta-llama/Llama-3.1-8B-Instruct",
]])

## 4 · Der KV-Cache

Der KV-Cache speichert Attention-Zwischenergebnisse. Er wächst linear mit Kontextlänge und gleichzeitigen Anfragen.


In [ ]:
# Vorgegebene Hilfsfunktion
def speicher_kv_cache(schichten, kv_koepfe, kopf_dimension, kontextlaenge,
                      batch=1, bits=16):
    """Speicher für den KV-Cache in GB. Die 2 steht für Key und Value."""
    bytes_pro_wert = bits / 8
    pro_token = 2 * schichten * kv_koepfe * kopf_dimension * bytes_pro_wert
    return pro_token * kontextlaenge * batch / GB

## 5 · Das VRAM-Budget

Für eine belastbare Abschätzung addieren wir Gewichte, KV-Cache und Overhead. Erst diese Summe wird mit dem verfügbaren VRAM verglichen.


In [ ]:
# ▶️ Die Annahme für Aktivierungen, Puffer und Runtime
OVERHEAD_ANTEIL = 0.15   # 15 % von Gewichten + KV-Cache
OVERHEAD_MIN = 1.0       # mindestens 1 GB, auch bei winzigen Modellen

print(f"Overhead = max({OVERHEAD_MIN} GB, {OVERHEAD_ANTEIL:.0%} × (Gewichte + KV-Cache))")

### 🛠️ Aufgabe 2 — Ein VRAM-Budget prüfen

Vervollständige `passt_auf(...)`: Addiere Gewichtsspeicher, KV-Cache und Overhead und vergleiche die Summe mit dem verfügbaren Speicher.

Führe danach den Selbsttest aus. Die ausgefüllte Variante steht in der Lösungsversion des Notebooks.


In [ ]:
def passt_auf(modell, hardware, bits=4, kontext=8192, batch=1):
    """Rechnet das VRAM-Budget eines Modells auf einer Karte durch."""
    # Im Speicher liegen immer alle Parameter — auch die Experten, die gerade
    # nicht gewählt sind.
    # TODO: Ersetze die nächste Zeile
    raise NotImplementedError("Aufgabe 2: passt_auf() implementieren")


In [ ]:
# ✅ Selbsttest
acht_b = MODELL_NACH_NAME["meta-llama/Llama-3.1-8B-Instruct"]
siebzig_b = MODELL_NACH_NAME["meta-llama/Llama-3.1-70B-Instruct"]
mixtral = MODELL_NACH_NAME["mistralai/Mixtral-8x7B-Instruct-v0.1"]
karte_4090 = KARTE_NACH_NAME["RTX 4090 (24 GB)"]
karte_a100 = KARTE_NACH_NAME["NVIDIA A100 SXM (80 GB)"]

u = passt_auf(acht_b, karte_4090, bits=16, kontext=8192)
assert abs(u["gewichte_gb"] - 16.06) < 0.01, f"Gewichte: {u['gewichte_gb']}"
assert abs(u["kv_gb"] - 1.074) < 0.01, f"KV-Cache: {u['kv_gb']}"
assert u["gesamt_gb"] > u["gewichte_gb"] + u["kv_gb"], "Der Overhead fehlt"
assert u["passt"] and abs(u["rest_gb"] - (24 - u["gesamt_gb"])) < 1e-9

v = passt_auf(siebzig_b, karte_4090, bits=4)
assert not v["passt"], "70B in INT4 sind 35 GB Gewichte — das passt nicht in 24 GB"
assert v["rest_gb"] < 0, "Wer nicht passt, hat negativen Rest"

w = passt_auf(siebzig_b, karte_a100, bits=4, kontext=8192, batch=8)
assert w["passt"], "70B in INT4, 8k Kontext, 8 Anfragen: das geht auf 80 GB"
assert not passt_auf(siebzig_b, karte_a100, bits=4, kontext=8192, batch=32)["passt"], \
    "32 gleichzeitige Anfragen sprengen dieselbe Karte"

# Bei MoE zählt für den Speicher die Gesamtgröße, nicht der aktive Anteil.
assert abs(passt_auf(mixtral, karte_a100)["gewichte_gb"] - 23.35) < 0.01

print("✅ Aufgabe 2 gelöst")
print()
for name, wert in passt_auf(siebzig_b, karte_a100, bits=4, kontext=8192, batch=8).items():
    print(f"  {name:<14} {wert if not isinstance(wert, float) else round(wert, 2)}")

In [ ]:
# ▶️ Die Matrix: alle Modelle gegen alle Karten, zwei Präzisionen
KONTEXT_MATRIX, BATCH_MATRIX = 8192, 1
FARBEN = matplotlib.colors.ListedColormap(["#fde2d4", "#d6efe9"])

fig, achsen = plt.subplots(1, 2, figsize=(14, 6.2), sharey=True)
bestanden = {}

for achse, (praezision, bits) in zip(achsen, [("FP16", 16), ("INT4", 4)]):
    urteile = np.zeros((len(MODELLE), len(HARDWARE)))
    for i, m in enumerate(MODELLE):
        for j, k in enumerate(HARDWARE):
            u = passt_auf(m, k, bits, KONTEXT_MATRIX, BATCH_MATRIX)
            urteile[i, j] = 1.0 if u["passt"] else 0.0
            achse.text(j, i, f"{u['rest_gb']:+.0f}", ha="center", va="center",
                       fontsize=8.5, fontweight="bold",
                       color=TEAL if u["passt"] else ORANGE)
    bestanden[praezision] = int(urteile.sum())

    achse.imshow(urteile, cmap=FARBEN, aspect="auto", vmin=0, vmax=1)
    achse.set_xticks(range(len(HARDWARE)))
    achse.set_xticklabels([k["name"] for k in HARDWARE], rotation=35, ha="right",
                          fontsize=8.5)
    achse.set_yticks(range(len(MODELLE)))
    achse.set_yticklabels([m["name"].split("/")[-1] for m in MODELLE], fontsize=9)
    achse.set_xticks(np.arange(-0.5, len(HARDWARE), 1), minor=True)
    achse.set_yticks(np.arange(-0.5, len(MODELLE), 1), minor=True)
    achse.grid(which="minor", color="white", linewidth=2)
    achse.grid(which="major", visible=False)
    achse.tick_params(which="minor", length=0)
    achse.set_title(f"{praezision} — {bestanden[praezision]} von "
                    f"{urteile.size} Kombinationen passen")

fig.suptitle(f"Freier Speicher in GB nach Gewichten, KV-Cache und Overhead — "
             f"{KONTEXT_MATRIX // 1024}k Kontext, {BATCH_MATRIX} Anfrage "
             f"(grün = passt)", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

print(f"FP16: {bestanden['FP16']} von 120   INT4: {bestanden['INT4']} von 120")

## 6 · Eine grobe Geschwindigkeitsgrenze

Beim Decoding werden Gewichte ständig aus dem Speicher gelesen. Speicherbandbreite geteilt durch aktive Modellgröße liefert nur eine theoretische Obergrenze.


In [ ]:
# ▶️ Die Obergrenze rechnen
def gelesen_je_token_gb(modell, bits=4):
    """Bytes, die für ein Token aus dem Speicher gelesen werden — bei MoE nur die aktiven."""
    aktiv = modell["parameter_aktiv"] or modell["parameter_gesamt"]
    return speicher_gewichte(aktiv, bits)


def obergrenze_tokens_pro_sekunde(modell, hardware, bits=4):
    """Speicherbandbreite ÷ gelesene Bytes je Token."""
    return hardware["speicherbandbreite_gb_s"] / gelesen_je_token_gb(modell, bits)


gezeigte_modelle = ["meta-llama/Llama-3.2-3B-Instruct", "meta-llama/Llama-3.1-8B-Instruct",
                    "mistralai/Mixtral-8x7B-Instruct-v0.1",
                    "meta-llama/Llama-3.1-70B-Instruct"]

zeige_tabelle([{
    "Hardware": k["name"],
    "Bandbreite (GB/s)": k["speicherbandbreite_gb_s"],
    **{n.split("/")[-1]: int(obergrenze_tokens_pro_sekunde(MODELL_NACH_NAME[n], k))
       for n in gezeigte_modelle},
} for k in HARDWARE])

print("Obergrenze in Tokens/s, alle Modelle in INT4, eine einzelne Anfrage.")
print("Die Zahl setzt voraus, dass das Modell auf die Karte passt — sonst ist sie")
print("hypothetisch. Mixtral steht hier vor Llama 70B, weil je Token nur die")
print("aktiven 12,9 Milliarden Parameter gelesen werden.")

In [ ]:
# ▶️ Welche Hardware steht hier, und welche Modelle liegen im lokalen Ollama?
HARDWARE_HIER = "MacBook Pro M4 Max (128 GB)"      # ← auf deinem Rechner anpassen

hier = KARTE_NACH_NAME[HARDWARE_HIER]
print(f"{hier['name']}: {hier['vram_gb']} GB, "
      f"{hier['speicherbandbreite_gb_s']} GB/s Speicherbandbreite")
print()

try:
    verfuegbar = {m.id.split(":")[0] for m in helfer.client.models.list().data}
    print("Ollama meldet:", ", ".join(sorted(verfuegbar)))
except Exception as fehler:
    verfuegbar = set()
    print(f"Ollama nicht erreichbar ({type(fehler).__name__}) — die Messung entfällt.")

In [ ]:
# ▶️ Messen. Der erste Aufruf lädt das Modell in den Speicher und zählt nicht.
CVE_TEXT = (
    "CVE-2026-3224 — CVSS 9.1. An unauthenticated attacker can send a crafted SAML "
    "assertion to the /sso/acs endpoint of NorthGate VPN Gateway 7.2 to 7.4 and obtain a "
    "valid administrator session. A public proof of concept exists. Patch 7.4.3 is "
    "available. Affected in our estate: 14 gateways, 3 of them reachable from the "
    "internet."
)
PROMPT = (f"{CVE_TEXT}\n\n"
          "You are briefing a CISO. Summarise this in about 120 words: exposure, "
          "exploitability, and the first mitigation step.")

# Drei Größen auf derselben Maschine: 1,0 GB, 2,0 GB und 9,6 GB im Speicher.
# Das Reasoning schaltet `helfer` selbst ab — sonst geht das Token-Budget ins
# Denken und `content` bleibt leer.
KANDIDATEN = ["qwen3.5:0.8b", "llama3.2", "gemma4"]
WIEDERHOLUNGEN = 3      # eine einzelne Messung streut zu stark

messungen = []
for tag in KANDIDATEN:
    if tag.split(":")[0] not in verfuegbar:
        print(f"{tag:<14} nicht installiert, übersprungen")
        continue

    # Der erste Aufruf lädt das Modell in den Speicher und zählt nicht mit.
    messe_anfrage("Warm up.", modell=tag, max_tokens=8)
    laeufe = [messe_anfrage(PROMPT, modell=tag, max_tokens=150)
              for _ in range(WIEDERHOLUNGEN)]

    messungen.append({
        "modell": tag,
        "ttft": statistics.median(l["ttft"] for l in laeufe),
        "tokens_pro_sekunde": statistics.median(l["tokens_pro_sekunde"] for l in laeufe),
        "prompt_tokens": laeufe[-1]["prompt_tokens"],
        "antwort_tokens": laeufe[-1]["antwort_tokens"],
        "antwort": laeufe[-1]["antwort"],
    })

    print(f"{tag:<14} "
          f"TTFT {messungen[-1]['ttft']:5.2f} s   "
          f"{messungen[-1]['prompt_tokens']:3d} Prompt-Tokens   "
          f"{messungen[-1]['antwort_tokens']:3d} Antwort-Tokens   "
          f"{messungen[-1]['tokens_pro_sekunde']:5.1f} Tokens/s (Median)")
    print("               einzelne Läufe: "
          + "   ".join(f"{l['ttft']:.2f} s / {l['tokens_pro_sekunde']:.1f} T/s"
                       for l in laeufe))

print()
if messungen:
    print(messungen[0]["antwort"][:320], "…")

In [ ]:
# ▶️ Messung gegen Obergrenze
zeilen = []
for messung in messungen:
    groesse = ollama_groesse_gb(messung["modell"])
    if groesse is None:
        continue
    grenze = hier["speicherbandbreite_gb_s"] / groesse
    zeilen.append({
        "Ollama-Modell": messung["modell"],
        "Größe im Speicher (GB)": groesse,
        "Bandbreite (GB/s)": hier["speicherbandbreite_gb_s"],
        "Obergrenze (Tokens/s)": grenze,
        "gemessen (Tokens/s)": messung["tokens_pro_sekunde"],
        "Ausnutzung": f"{messung['tokens_pro_sekunde'] / grenze * 100:.0f} %",
        "TTFT (s)": messung["ttft"],
    })

zeige_tabelle(zeilen)

In [ ]:
# ▶️ Dasselbe als Bild: die Obergrenze als Kurve, die Messungen als Punkte
if zeilen:
    groessen = np.logspace(np.log10(0.5), np.log10(40), 100)
    kurve = hier["speicherbandbreite_gb_s"] / groessen

    fig, achse = plt.subplots(figsize=(9, 4.4))
    achse.plot(groessen, kurve, color=GRAU, linewidth=1.6,
               label=f"Obergrenze = {hier['speicherbandbreite_gb_s']} GB/s ÷ Modellgröße")

    x = [z["Größe im Speicher (GB)"] for z in zeilen]
    y = [z["gemessen (Tokens/s)"] for z in zeilen]
    achse.scatter(x, y, s=90, color=ORANGE, zorder=3, label="gemessen")
    for z in zeilen:
        achse.annotate(f"  {z['Ollama-Modell']}  ({z['Ausnutzung']})",
                       (z["Größe im Speicher (GB)"], z["gemessen (Tokens/s)"]),
                       fontsize=9, color=BLAU, va="center")
        achse.plot([z["Größe im Speicher (GB)"]] * 2,
                   [z["gemessen (Tokens/s)"], z["Obergrenze (Tokens/s)"]],
                   color=ORANGE, linewidth=1.0, linestyle=":", zorder=2)

    achse.set_xscale("log")
    achse.set_yscale("log")
    achse.set_xlabel("Größe des Modells im Speicher (GB)")
    achse.set_ylabel("Tokens je Sekunde, eine Anfrage")
    achse.set_title(f"Obergrenze und Messung auf {hier['name']}")
    achse.legend(frameon=False, fontsize=9, loc="lower left")
    plt.tight_layout()
    plt.show()

## 7 · Deployment entscheiden

Eine Entscheidung verbindet Modellquelle, Betriebsort, Runtime und Zugriff. Speicher ist eine harte Grenze; Geschwindigkeit und Governance sind weitere Anforderungen.


In [ ]:
# ▶️ Die sechs Szenarien
import textwrap

for s in SZENARIEN:
    print(f"[{s['id']}] {s['titel']}")
    print(f"      Hardware: {s['hardware'] or 'keine'}   "
          f"Kontext: {s['kontext']}   gleichzeitige Anfragen: {s['batch']}")
    print(textwrap.fill(s["text"], width=100,
                        initial_indent="      ", subsequent_indent="      "))
    print()

In [ ]:
# ▶️ Die Prüffunktion — sie vergleicht mit den hinterlegten Sollantworten
FRAGEN = ("quelle", "ort", "runtime", "zugriff")


def pruefe_entscheidungen(entscheidungen, laut=True):
    """Vergleicht die vier Antworten je Szenario und rechnet die Modellwahl nach.

    Rückgabe: Zahl der Szenarien, bei denen alle vier Antworten stimmen und das
    gewählte Modell auf die genannte Hardware passt.
    """
    vollstaendig = 0

    for s in SZENARIEN:
        meine = entscheidungen.get(s["id"], {})
        soll = s["loesung"]
        falsch = [f for f in FRAGEN if meine.get(f) != soll[f]]

        anmerkung = ""
        modell_ok = True
        if s["hardware"]:
            modell = MODELL_NACH_NAME.get(meine.get("modell"))
            if modell is None:
                modell_ok = False
                anmerkung = f"Modell {meine.get('modell')!r} steht nicht in den Model Cards"
            else:
                urteil = passt_auf(modell, KARTE_NACH_NAME[s["hardware"]],
                                   meine.get("bits", 4), s["kontext"], s["batch"])
                modell_ok = urteil["passt"]
                anmerkung = (f"{modell['name'].split('/')[-1]} @ {meine.get('bits', 4)} Bit: "
                             f"{urteil['gesamt_gb']:.1f} von {urteil['vram_gb']} GB, "
                             f"{'passt' if modell_ok else 'passt nicht'}")

        if not falsch and modell_ok:
            vollstaendig += 1

        if laut:
            zeichen = "✓" if not falsch and modell_ok else "✗"
            print(f"{zeichen} [{s['id']}] {s['titel']}")
            for frage in FRAGEN:
                marke = " " if frage not in falsch else "→"
                soll_text = "" if frage not in falsch else f"   (richtig: {soll[frage]})"
                print(f"    {marke} {frage:<8} {meine.get(frage, '—')}{soll_text}")
            if anmerkung:
                print(f"      Modell   {anmerkung}")
            print()

    return vollstaendig

In [ ]:
# Vorgegebene Hilfsfunktion
ENTSCHEIDUNGEN = {
    # Prototyp, ein Mensch, nichts verlässt das Gerät, späterer Umzug auf einen Server.
    "s1": {"quelle": "hub", "ort": "lokal", "runtime": "ollama", "zugriff": "lokale-api",
           "modell": "meta-llama/Llama-3.1-8B-Instruct", "bits": 4},
    # 40 Nutzende, Daten dürfen das Haus nicht verlassen, A100 ist da.
    "s2": {"quelle": "hub", "ort": "on-premise", "runtime": "vllm", "zugriff": "interne-api",
           "modell": "meta-llama/Llama-3.1-70B-Instruct", "bits": 4},
    # Eigenes Fine-Tuning, keine eigene Hardware, aber ein privater Cloud-Mandant.
    "s3": {"quelle": "eigenes-fine-tuning", "ort": "cloud", "runtime": "vllm",
           "zugriff": "interne-api",
           "modell": "meta-llama/Llama-3.1-8B-Instruct", "bits": 4},
    # Batch-Job über Nacht, niemand wartet, gebraucht werden die Logits.
    "s4": {"quelle": "hub", "ort": "lokal", "runtime": "eigenes-skript",
           "zugriff": "in-process",
           "modell": "Qwen/Qwen2.5-0.5B-Instruct", "bits": 8},
    # Öffentliche Texte, keine Hardware, keine Betriebsmannschaft, höchste Qualität.
    "s5": {"quelle": "provider-modell", "ort": "provider-api", "runtime": "provider",
           "zugriff": "externe-api",
           "modell": None, "bits": None},
    # Vergleich ohne Python, 8 GB Laptop-GPU, eine Anfrage zur Zeit.
    "s6": {"quelle": "hub", "ort": "lokal", "runtime": "lm-studio", "zugriff": "lokale-api",
           "modell": "meta-llama/Llama-3.2-3B-Instruct", "bits": 4},
}

## Fazit

Du kannst Model-Card-Werte in ein Speicherbudget übersetzen und ein Modell gegen konkrete Hardware prüfen. Merksatz: **Nicht die Parameterzahl allein entscheidet, sondern Gewichte plus Laufzeitspeicher plus Reserve.**
